In [1]:
agent_request: Dict[str, Any] = {
    "run_id": "20260426_140845",
    "created_at": "2026-04-26T14:08:45",
    "user_request": "요구사항-기능-UI 간 정합성과 누락 기능을 우선 확인해줘.",
    "scenario_order": [
        "basic_quality",
        "traceability",
        "ui_match",
        "coverage",
    ],
    "document_count": 3,
    "request_path": "C:/Users/DJ/Documents/GitHub/SKAX-MasterProject/projects/data/intake/20260426_140845/agent_request.json",
    "documents": [
        {
            "document_key": "requirement_definition",
            "document_label": "요구사항 정의서",
            "file_name": "요구사항정의서.xlsx",
            "file_extension": ".xlsx",
            "mime_type": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "size_bytes": 13049,
            "sha256": "85e8cc17b3f4d45f65abdd6502aece6a43b20e0562ee9984c7e5dc8080d1aa2e",
            "saved_path": "C:/Users/DJ/Documents/GitHub/SKAX-MasterProject/projects/data/intake/20260426_140845/requirement_definition__uploaded_file.xlsx",
            "content_summary": {
                "parser": "builtin_xlsx_xml",
                "parser_status": "success",
                "sheet_count": 1,
                "sheet_names": ["Sheet1"],
                "sheets": [
                    {
                        "sheet_name": "Sheet1",
                        "file_name": "요구사항 정의서",
                        "columns": [
                            "시스템\n(Application)",
                            "업무그룹",
                            "요구사항 ID",
                            "요구사항명",
                            "요청자\n(요구사항 Owner)",
                            "상태",
                        ],
                        "data": [
                            {
                                "시스템\n(Application)": "모바일쇼핑",
                                "업무그룹": "로그인",
                                "요구사항 ID": "REQ_001",
                                "요구사항명": "로그인 화면_ID/Passwd 인증, 생체인증",
                                "요청자\n(요구사항 Owner)": "홍길동",
                                "상태": "수정",
                            },
                            {
                                "시스템\n(Application)": "",
                                "업무그룹": "검색",
                                "요구사항 ID": "REQ_002",
                                "요구사항명": "검색화면_상단 검색어 입력 기능",
                                "요청자\n(요구사항 Owner)": "",
                                "상태": "최초",
                            },
                        ],
                        "row_count": 2,
                    }
                ],
            },
        },
        {
            "document_key": "feature_definition",
            "document_label": "기능 정의서",
            "file_name": "기능정의서.xlsx",
            "file_extension": ".xlsx",
            "mime_type": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "size_bytes": 11647,
            "sha256": "0ad4d53bc46a1f562a9f16f5f3c8b2d6e1322f2c6881523da138d329705ad25c",
            "saved_path": "C:/Users/DJ/Documents/GitHub/SKAX-MasterProject/projects/data/intake/20260426_140845/feature_definition__uploaded_file.xlsx",
            "content_summary": {
                "parser": "builtin_xlsx_xml",
                "parser_status": "success",
                "sheet_count": 1,
                "sheet_names": ["Sheet1"],
                "sheets": [
                    {
                        "sheet_name": "Sheet1",
                        "file_name": "기능 정의서",
                        "columns": [
                            "시스템\n(Application)",
                            "요구사항 ID",
                            "기능ID",
                            "기능명",
                            "화면ID",
                        ],
                        "data": [
                            {
                                "시스템\n(Application)": "모바일쇼핑",
                                "요구사항 ID": "REQ_001",
                                "기능ID": "F-01",
                                "기능명": "로그인 화면_ID/Passwd 인증, 생체인증",
                                "화면ID": "LOGIN-01",
                            }
                        ],
                        "row_count": 1,
                    }
                ],
            },
        },
        {
            "document_key": "ui_design",
            "document_label": "UI 설계서",
            "file_name": "UI설계서.xlsx",
            "file_extension": ".xlsx",
            "mime_type": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "size_bytes": 13049,
            "sha256": "85e8cc17b3f4d45f65abdd6502aece6a43b20e0562ee9984c7e5dc8080d1aa2e",
            "saved_path": "C:/Users/DJ/Documents/GitHub/SKAX-MasterProject/projects/data/intake/20260426_140845/ui_design__UI.xlsx",
            "content_summary": {
                "parser": "builtin_xlsx_xml",
                "parser_status": "success",
                "sheet_count": 1,
                "sheet_names": ["Sheet1"],
                "sheets": [
                    {
                        "sheet_name": "Sheet1",
                        "file_name": "UI 설계서",
                        "columns": [
                            "화면ID",
                            "화면명",
                            "설명",
                        ],
                        "data": [
                            {
                                "화면ID": "LOGIN-01",
                                "화면명": "로그인 화면",
                                "설명": "ID/PW 입력 및 로그인 버튼 제공",
                            }
                        ],
                        "row_count": 1,
                    }
                ],
            },
        },
    ],
}


In [9]:
from langchain_openai  import AzureChatOpenAI
from utils.config_loader import load_config

# --- [LLM 설정] ---
LLM_KEY = load_config("LLM", "KEY")
LLM_MODEL = load_config("LLM", "MODEL")
LLM_ENDPOINT = load_config("LLM", "ENDPOINT")
LLM_VERSION = load_config("LLM", "VERSION")

LLM = AzureChatOpenAI(
    deployment_name = LLM_MODEL,
    azure_endpoint = LLM_ENDPOINT,
    api_key = LLM_KEY,
    api_version= LLM_VERSION,
    temperature = 0.2
)


In [39]:
"""LangChain DeepAgents 기준의 최소 Orchestrator 구성."""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field
from langchain_openai  import AzureChatOpenAI

from utils.config_loader import load_config
from deepagents import create_deep_agent

try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path(os.getcwd())
DATA_ROOT = PROJECT_ROOT / "data" / "subagents"
DEFAULT_MODEL = os.getenv("DEEPAGENT_MODEL", "google_genai:gemini-3.1-pro-preview")

# ---[ LLM 설정 ]---
LLM_KEY = load_config("LLM", "KEY")
LLM_MODEL = load_config("LLM", "MODEL")
LLM_ENDPOINT = load_config("LLM", "ENDPOINT")
LLM_VERSION = load_config("LLM", "VERSION")
LLM = AzureChatOpenAI(
    deployment_name = LLM_MODEL,
    azure_endpoint = LLM_ENDPOINT,
    api_key = LLM_KEY,
    api_version= LLM_VERSION,
    temperature = 0.2
)
# ---[ LLM 설정 ]---

# ---[ Agent Return Format ]---
class SubagentReport(BaseModel):
    """서브에이전트가 반환하는 최소 구조."""

    scenario_key: str = Field(description="현재 시나리오 키")
    summary: str = Field(description="서브에이전트 요약")
    score: Optional[int] = Field(default=None, description="0~100 점수")
    findings: List[str] = Field(default_factory=list, description="핵심 이슈 목록")
    warnings: List[str] = Field(default_factory=list, description="주의사항 목록")
    recommendations: List[str] = Field(default_factory=list, description="권장 조치 목록")
    artifact_path: Optional[str] = Field(default=None, description="저장 경로")
class ScenarioReport(BaseModel):
    """최종 보고서의 시나리오 단위 구조."""

    scenario_key: str = Field(description="시나리오 키")
    scenario_label: str = Field(description="표시용 시나리오 이름")
    status: str = Field(description="통과, 검토 권장, 보완 필요 중 하나")
    score: int = Field(description="시나리오 점수", ge=0, le=100)
    summary: str = Field(description="시나리오 요약")
    findings: List[str] = Field(default_factory=list, description="주요 이슈")
    warnings: List[str] = Field(default_factory=list, description="주의사항")
    recommendations: List[str] = Field(default_factory=list, description="권장 조치")
class FinalReviewReport(BaseModel):
    """메인 DeepAgent의 최종 구조화 응답."""

    run_id: str = Field(description="실행 식별자")
    summary: str = Field(description="전체 점검 요약")
    overall_score: int = Field(description="통합 점수", ge=0, le=100)
    blocked_scenarios: List[str] = Field(default_factory=list, description="보완 필요 시나리오")
    scenario_order: List[str] = Field(default_factory=list, description="실행 시나리오 순서")
    scenario_results: List[ScenarioReport] = Field(default_factory=list, description="시나리오별 결과")
    priority_actions: List[str] = Field(default_factory=list, description="우선순위 액션")
# ---[ Agent Return Format ]---

# ---[ SubAgent Static Setting ]---
SUBAGENT_SPECS = [
    {
        "name": "guide-agent",
        "description": "시나리오 기준, 필수 문서, 사전 체크를 짧게 정리한다.",
        "skills_key": "guide_agent",
        "tools_key": "guide",
        "system_prompt": (
            "너는 Guide Agent다. 시나리오 정의와 현재 문서 현황을 먼저 확인하고, "
            "필수 문서 누락 여부와 체크리스트를 짧게 정리하라."
        ),
        "response_format": SubagentReport,
    },
    {
        "name": "qa-agent",
        "description": "문서가 애매할 때 추가 확인 질문을 짧게 정리한다.",
        "skills_key": "qa_agent",
        "tools_key": "shared",
        "system_prompt": (
            "너는 QA Agent다. 문서 행 수, 파싱 상태, 샘플 행을 보고 "
            "추가로 확인해야 할 질문만 간단히 정리하라."
        ),
        "response_format": SubagentReport,
    },
    {
        "name": "validation-agent",
        "description": "basic_quality와 traceability를 룰 기반으로 점검한다.",
        "skills_key": "validation_agent",
        "tools_key": "validation",
        "system_prompt": (
            "너는 Validation Agent다. basic_quality와 traceability 시나리오에서는 "
            "제공된 review tool을 사용해 findings, warnings, score를 정리하라."
        ),
        "response_format": SubagentReport,
    },
    {
        "name": "review-agent",
        "description": "ui_match와 coverage를 심층 검토한다.",
        "skills_key": "deep_review_agent",
        "tools_key": "review",
        "system_prompt": (
            "너는 Review Agent다. ui_match와 coverage 시나리오에서 "
            "문서 간 불일치와 누락 위험을 정리하라."
        ),
        "response_format": SubagentReport,
    },
    {
        "name": "improvement-agent",
        "description": "이슈를 우선순위 중심 개선 액션으로 바꾼다.",
        "skills_key": "improvement_agent",
        "tools_key": "improvement",
        "system_prompt": (
            "너는 Improvement Agent다. findings와 warnings를 받아 "
            "실행 가능한 개선 액션만 간단히 정리하라."
        ),
        "response_format": SubagentReport,
    },
    {
        "name": "report-agent",
        "description": "시나리오별 결과를 최종 보고서로 통합한다.",
        "skills_key": "report_agent",
        "tools_key": "shared",
        "system_prompt": (
            "너는 Report Agent다. 시나리오별 결과를 통합해 전체 점수, "
            "보완 필요 시나리오, 우선순위 액션이 보이는 최종 보고서를 작성하라."
        ),
        "response_format": FinalReviewReport,
    },
]
# ---[ SubAgent Static Setting ]---

def create_orchestrator_agent(agent_request: Dict[str, Any]):
    """공식 DeepAgents 형태로 Orchestrator를 생성합니다."""
    toolset = build_toolset(agent_request)

    subagents = []
    for spec in SUBAGENT_SPECS: # 서브 에이전트 목록 정의 
        subagents.append(
            {
                "name": spec["name"],
                "description": spec["description"],
                "system_prompt": spec["system_prompt"],
                "tools": toolset[spec["tools_key"]],
                "skills": toolset["skills"][spec["skills_key"]],
                "response_format": spec["response_format"],
            }
        )

    return create_deep_agent(
        model = LLM,
        tools=toolset["shared"],
        system_prompt=build_system_prompt(agent_request),
        subagents=subagents,
        skills=toolset["skills"]["orchestrator_agent"],
        response_format=FinalReviewReport,
        name="deliverable-review-orchestrator",
    )

def run_orchestrator(agent_request: Dict[str, Any]) -> Dict[str, Any]:
    """DeepAgents 실행 후 결과를 반환합니다."""
    run_id = agent_request.get("run_id", "manual_run") 
    agent = create_orchestrator_agent(agent_request)
    task = build_task_prompt(agent_request)  

    ## ---[ Agent 실행 ]---
    raw_result = agent.invoke({"messages": [{"role": "user", "content": task}]})

    ## ---[ 결과 처리 ]---
    final_report = normalize_report(
        run_id=run_id,
        scenario_order=agent_request.get("scenario_order", []),
        structured_response=raw_result.get("structured_response"),
        fallback_text=extract_last_message(raw_result),
    )
    save_json(DATA_ROOT / run_id / "final_report.json", final_report)

    result = {
        "status": "completed",
        "mode": "langchain_deepagents",
        "message": "LangChain DeepAgents Orchestrator 실행이 완료되었습니다.",
        "run_id": run_id,
        "model": LLM.profile['name'],
        "document_catalog": get_document_catalog_data(agent_request.get("documents", [])),
        "final_report": final_report,
        "raw_last_message": extract_last_message(raw_result),
    }
    return save_orchestrator_result(run_id, result)


def build_system_prompt(agent_request: Dict[str, Any]) -> str:
    """메인 Orchestrator Agent용 시스템 프롬프트."""
    scenario_order = ", ".join(agent_request.get("scenario_order", [])) or "basic_quality"
    lines = []
    for document in get_document_catalog_data(agent_request.get("documents", [])):
        lines.append(
            f"- {document['document_label']} / rows={document['row_count']} / parser={document['parser_status']}"
        )
    document_summary = "\n".join(lines) if lines else "- 업로드 문서 없음"

    return f"""너는 프로젝트 산출물 점검을 총괄하는 Orchestrator Agent다.

반드시 LangChain DeepAgents 방식으로 동작하고, 세부 분석은 직접 오래 하지 말고 subagent에 위임하라.

현재 시나리오 순서:
{scenario_order}

현재 문서 현황:
{document_summary}

운영 규칙:
1. 시작 시 문서 상태를 확인한다.
2. 시나리오는 순차 실행한다.
3. basic_quality, traceability는 validation-agent를 우선 활용한다.
4. ui_match, coverage는 review-agent를 우선 활용한다.
5. 필요한 경우 guide-agent, qa-agent, improvement-agent, report-agent를 호출한다.
6. 최종 응답은 반드시 구조화된 보고서로 반환한다.
"""

def build_task_prompt(agent_request: Dict[str, Any]) -> str:
    """실행 시 user message로 넘길 태스크 프롬프트."""
    ordered = "\n".join(
        f"{index}. {scenario_key}"
        for index, scenario_key in enumerate(agent_request.get("scenario_order", []), start=1)
    ) or "1. basic_quality"

    return f"""run_id `{agent_request.get('run_id', 'manual_run')}` 에 대한 산출물 점검을 수행하라.

추가 요청:
{agent_request.get('user_request') or '추가 요청 없음'}

실행 시나리오 순서:
{ordered}

반드시 시나리오별 요약, 점수, findings, warnings, recommendations 를 정리하고,
마지막에는 전체 점수와 우선순위 액션을 포함한 최종 보고서를 반환하라.
"""

def normalize_report(
    run_id: str,
    scenario_order: List[str],
    structured_response: Any,
    fallback_text: str,
) -> Dict[str, Any]:
    """structured_response를 저장 가능한 dict로 바꿉니다."""
    if structured_response is None:
        return {
            "run_id": run_id,
            "summary": fallback_text or "구조화 응답이 비어 있습니다.",
            "overall_score": 0,
            "blocked_scenarios": [],
            "scenario_order": list(scenario_order),
            "scenario_results": [],
            "priority_actions": [],
        }

    if hasattr(structured_response, "model_dump"):
        report = structured_response.model_dump()
    elif isinstance(structured_response, dict):
        report = dict(structured_response)
    else:
        report = json.loads(json.dumps(structured_response, ensure_ascii=False, default=str))

    report.setdefault("run_id", run_id)
    report.setdefault("summary", fallback_text or "점검 보고서가 생성되었습니다.")
    report.setdefault("overall_score", 0)
    report.setdefault("blocked_scenarios", [])
    report.setdefault("scenario_order", list(scenario_order))
    report.setdefault("scenario_results", [])
    report.setdefault("priority_actions", [])
    return report


def extract_last_message(raw_result: Dict[str, Any]) -> str:
    """DeepAgents 실행 결과에서 마지막 메시지를 꺼냅니다."""
    messages = raw_result.get("messages", [])
    if not messages:
        return ""
    content = getattr(messages[-1], "content", "")
    return content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, default=str)


def save_orchestrator_result(run_id: str, result: Dict[str, Any]) -> Dict[str, Any]:
    """Orchestrator 결과를 저장하고 data_path를 붙입니다."""
    file_path = DATA_ROOT / run_id / "orchestrator_output.json"
    save_json(file_path, result)
    result["data_path"] = str(file_path)
    save_json(file_path, result)
    return result



In [52]:
SCENARIOS = {
    "basic_quality": {  # 1단계 기초 품질 점검
        "label": "기초 품질 점검",  # 이름
        "description": "- 형식 오류, 필수 항목 누락, 용어 불일치를 먼저 점검합니다.",  # 설명
        "required_files": ["요구사항 정의서", "기능 정의서", "UI 설계서"],  # 필수 문서
        "checks": [  # 주요 점검 항목
            "ID 형식 규칙",  # 식별자 규칙 확인
            "필수 항목 누락",  # 비어 있는 필수 값 점검
            "용어 표준화 여부",  # 표준 용어 사용 여부
            "기본 체크리스트 통과 여부",  # 기본 품질 기준 충족 여부
        ],
        "outputs": ["점검 점수", "오류 목록", "수정 가이드"],  # 출력 형식
    },
    "traceability": {  # 2단계 문서 연결성 점검
        "label": "문서 연결성 점검",  # 이름
        "description": "- 요구사항, 기능, UI 문서가 같은 흐름으로 연결되는지 확인합니다.",  # 설명
        "required_files": ["요구사항 정의서", "기능 정의서", "UI 설계서"],  # 필수 문서
        "checks": [  # 주요 점검 항목
            "요구사항-기능 연결",  # 요구사항과 기능 연결 확인
            "기능-UI 연결",  # 기능과 UI 연결 확인
            "미연결 ID 탐지",  # 연결되지 않은 식별자 탐지
            "정의 외 항목 탐지",  # 근거 없는 항목 탐지
        ],
        "outputs": ["정합성 커버리지", "미연결 목록", "보정 우선순위"],  # 출력 형식
    },
    "ui_match": {  # 3단계 기능-화면 일치 점검
        "label": "기능-화면 일치 점검",  # 이름
        "description": "- 문서에 정의된 기능이 화면 설계와 맞물리는지 점검합니다.",  # 설명
        "required_files": ["기능 정의서", "UI 설계서"],  # 필수 문서
        "checks": [  # 주요 점검 항목
            "기능과 버튼 대응",  # 기능과 UI 액션 연결 확인
            "목록/상세 흐름 일치",  # 화면 흐름 일관성 확인
            "누락 화면 요소",  # 빠진 UI 요소 탐지
            "불필요 UI 존재 여부",  # 필요 없는 UI 존재 여부
        ],
        "outputs": ["불일치 항목", "개선 제안", "검증 메모"],  # 출력 형식
    },
    "coverage": {  # 4단계 기능 완전성 분석
        "label": "기능 완전성 분석",  # 이름
        "description": "- 요구사항 대비 기능 정의가 충분한지, 누락과 과잉을 함께 분석합니다.",  # 설명
        "required_files": ["요구사항 정의서", "기능 정의서"],  # 필수 문서
        "checks": [  # 주요 점검 항목
            "기능 누락",  # 빠진 기능 점검
            "과잉 기능",  # 필요 이상 기능 점검
            "세부 기능 분해 적절성",  # 기능 분해 수준 확인
            "추가 정의 필요 영역",  # 보완이 필요한 영역 확인
        ],
        "outputs": ["완전성 점수", "Gap 분석", "보완 제안"],  # 출력 형식
    },
}
"""DeepAgents에서 사용하는 최소 review tool 모음."""

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

from langchain_core.tools import tool

from ui.service_data import get_scenario_config


try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path(os.getcwd())
    
DATA_ROOT = PROJECT_ROOT / "data" / "subagents"
SKILLS_ROOT = PROJECT_ROOT / "skills"
DOCUMENT_LABELS = {
    "requirement_definition": "요구사항 정의서",
    "feature_definition": "기능 정의서",
    "ui_design": "UI 설계서",
}
TERM_GROUPS = [("조회", "검색"), ("사용자", "회원"), ("삭제", "제거")]


def build_toolset(agent_request: Dict[str, Any]) -> Dict[str, Any]:
    """현재 요청 기준으로 DeepAgents tool registry를 만듭니다."""
    run_id = agent_request.get("run_id", "manual_run")
    documents = agent_request.get("documents", [])

    @tool("get_document_catalog")
    def get_document_catalog_tool() -> Dict[str, Any]:
        """현재 업로드된 문서의 파싱 상태와 행 수를 반환합니다."""
        return {"documents": get_document_catalog_data(documents)}

    @tool("get_document_preview")
    def get_document_preview_tool(document_key: str, max_rows: int = 3) -> Dict[str, Any]:
        """문서 컬럼과 샘플 행을 미리 확인합니다."""
        document = get_document(documents, document_key)
        if document is None:
            return {"document_key": document_key, "status": "not_found"}
        rows = get_rows(document)
        return {
            "document_key": document_key,
            "document_label": document.get("document_label", ""),
            "columns": get_columns(document),
            "row_count": len(rows),
            "preview_rows": rows[: max(1, min(max_rows, 10))],
        }

    @tool("get_scenario_definition")
    def get_scenario_definition_tool(scenario_key: str) -> Dict[str, Any]:
        """시나리오 설명, 필수 문서, 점검 항목을 반환합니다."""
        return {"scenario_key": scenario_key, **get_scenario_config(scenario_key)}

    @tool("run_basic_quality_review")
    def run_basic_quality_review_tool() -> Dict[str, Any]:
        """기초 품질 점검을 수행합니다."""
        return review_basic_quality(documents)

    @tool("run_traceability_review")
    def run_traceability_review_tool() -> Dict[str, Any]:
        """요구사항-기능-UI 연결 구조를 점검합니다."""
        return review_traceability(documents)

    @tool("run_ui_match_review")
    def run_ui_match_review_tool() -> Dict[str, Any]:
        """기능과 UI 간 일치성을 점검합니다."""
        return review_ui_match(documents)

    @tool("run_coverage_review")
    def run_coverage_review_tool() -> Dict[str, Any]:
        """요구사항 대비 기능 완전성을 점검합니다."""
        return review_coverage(documents)

    @tool("build_improvement_actions")
    def build_improvement_actions_tool(
        scenario_key: str,
        findings: List[str],
        warnings: List[str],
    ) -> List[str]:
        """findings와 warnings를 기반으로 개선 액션을 만듭니다."""
        return make_actions(scenario_key, findings, warnings)

    @tool("persist_subagent_output")
    def persist_subagent_output_tool(scenario_key: str, agent_name: str, payload_json: str) -> str:
        """서브에이전트 결과를 JSON 파일로 저장합니다."""
        payload = parse_json(payload_json)
        file_path = DATA_ROOT / run_id / scenario_key / f"{sanitize_name(agent_name)}.json"
        save_json(file_path, payload)
        return str(file_path)

    shared = [
        get_document_catalog_tool,
        get_document_preview_tool,
        get_scenario_definition_tool,
        persist_subagent_output_tool,
    ]
    return {
        "shared": shared,
        "guide": shared,
        "validation": [
            get_scenario_definition_tool,
            run_basic_quality_review_tool,
            run_traceability_review_tool,
            persist_subagent_output_tool,
        ],
        "review": [
            get_scenario_definition_tool,
            run_ui_match_review_tool,
            run_coverage_review_tool,
            persist_subagent_output_tool,
        ],
        "improvement": [
            get_scenario_definition_tool,
            build_improvement_actions_tool,
            persist_subagent_output_tool,
        ],
        "skills": {
            "orchestrator_agent": [str(SKILLS_ROOT / "orchestrator_agent")],
            "guide_agent": [str(SKILLS_ROOT / "guide_agent")],
            "qa_agent": [str(SKILLS_ROOT / "qa_agent")],
            "validation_agent": [str(SKILLS_ROOT / "validation_agent")],
            "deep_review_agent": [str(SKILLS_ROOT / "deep_review_agent")],
            "improvement_agent": [str(SKILLS_ROOT / "improvement_agent")],
            "report_agent": [str(SKILLS_ROOT / "report_agent")],
        },
    }


def get_document_catalog_data(documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """문서별 파서 상태와 행 수만 간단히 요약합니다."""
    return [
        {
            "document_key": document.get("document_key", ""),
            "document_label": DOCUMENT_LABELS.get(document.get("document_key", ""), document.get("document_label", "")),
            "file_name": document.get("file_name", ""),
            "parser_status": document.get("content_summary", {}).get("parser_status", "unknown"),
            "row_count": len(get_rows(document)),
        }
        for document in documents
    ]


def review_basic_quality(documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """기초 품질 점검 휴리스틱."""
    findings: List[str] = []
    warnings: List[str] = []

    for document_key, document_label in DOCUMENT_LABELS.items():
        document = get_document(documents, document_key)
        if document is None:
            findings.append(f"{document_label} 문서가 없습니다.")
            continue

        rows = get_rows(document)
        columns = get_columns(document)
        if document.get("content_summary", {}).get("parser_status") != "success":
            findings.append(f"{document_label} 문서 파싱이 정상 완료되지 않았습니다.")
            continue
        if not rows:
            findings.append(f"{document_label}에서 읽힌 데이터 행이 없습니다.")
            continue

        id_column = find_column(rows, ["id"])
        name_column = find_column(rows, ["명"])
        if id_column:
            empty_ids = [str(index) for index, row in enumerate(rows, start=1) if not str(row.get(id_column, "")).strip()]
            invalid_ids = [
                str(row.get(id_column, ""))
                for row in rows
                if str(row.get(id_column, "")).strip() and not re.match(r"^[A-Za-z][A-Za-z0-9_-]+$", str(row.get(id_column, "")).strip())
            ]
            if empty_ids:
                findings.append(f"{document_label}의 식별자 누락 행이 있습니다. 예: {', '.join(empty_ids[:3])}행")
            if invalid_ids:
                findings.append(f"{document_label}의 식별자 형식이 일정하지 않습니다. 예: {', '.join(invalid_ids[:3])}")
        if id_column and name_column:
            missing_rows = [
                str(index)
                for index, row in enumerate(rows, start=1)
                if not str(row.get(id_column, "")).strip() or not str(row.get(name_column, "")).strip()
            ]
            if missing_rows:
                warnings.append(f"{document_label}의 필수값 누락 행이 있습니다. 예: {', '.join(missing_rows[:3])}행")
        if document_key == "ui_design" and not find_column(rows, ["화면id"]):
            warnings.append("UI 설계서에서 화면ID 컬럼을 찾지 못했습니다.")
        if not columns:
            warnings.append(f"{document_label}의 컬럼 정보를 읽지 못했습니다.")

    combined_text = " ".join(" ".join(str(value) for row in get_rows(doc) for value in row.values()) for doc in documents)
    for left, right in TERM_GROUPS:
        if left in combined_text and right in combined_text:
            warnings.append(f"`{left}`와 `{right}` 용어가 혼용되어 있습니다.")

    score = max(0, 100 - len(findings) * 12 - len(warnings) * 4)
    return {
        "scenario_key": "basic_quality",
        "summary": "형식, 필수값, 용어 기준으로 기초 품질을 점검했습니다.",
        "score": score,
        "findings": findings,
        "warnings": warnings,
        "recommendations": make_actions("basic_quality", findings, warnings),
    }


def review_traceability(documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """요구사항-기능-UI 연결 구조 점검."""
    requirement_rows = get_rows(get_document(documents, "requirement_definition"))
    feature_rows = get_rows(get_document(documents, "feature_definition"))
    ui_rows = get_rows(get_document(documents, "ui_design"))

    req_id_col = find_column(requirement_rows, ["요구사항 id"])
    feat_req_col = find_column(feature_rows, ["요구사항 id"])
    feat_screen_col = find_column(feature_rows, ["화면id"])
    ui_screen_col = find_column(ui_rows, ["화면id"])

    req_ids = set(non_empty_values(requirement_rows, req_id_col))
    feat_req_ids = set(non_empty_values(feature_rows, feat_req_col))
    feat_screen_ids = set(non_empty_values(feature_rows, feat_screen_col))
    ui_screen_ids = set(non_empty_values(ui_rows, ui_screen_col))

    findings: List[str] = []
    warnings: List[str] = []

    if not req_id_col:
        findings.append("요구사항 정의서에서 요구사항 ID 컬럼을 찾지 못했습니다.")
    if not feat_req_col:
        findings.append("기능 정의서에서 요구사항 ID 컬럼을 찾지 못했습니다.")
    if missing_values(req_ids, feat_req_ids):
        findings.append(f"요구사항 중 기능 정의로 연결되지 않은 ID가 있습니다. 예: {', '.join(missing_values(req_ids, feat_req_ids)[:3])}")
    if feat_screen_ids and not ui_screen_col:
        findings.append("UI 설계서에서 화면ID 컬럼을 찾지 못했습니다.")
    if feat_screen_ids and ui_screen_ids:
        gap = missing_values(feat_screen_ids, ui_screen_ids)
        if gap:
            findings.append(f"기능 정의서의 화면ID가 UI 설계서와 연결되지 않습니다. 예: {', '.join(gap[:3])}")
    if ui_rows and req_ids and feat_req_ids != req_ids:
        warnings.append("요구사항-기능-UI 연결이 일부 구간에서 끊겨 있을 수 있습니다.")

    score = max(0, 100 - len(findings) * 12 - len(warnings) * 4)
    return {
        "scenario_key": "traceability",
        "summary": "요구사항, 기능, UI 간 ID 연결 구조를 점검했습니다.",
        "score": score,
        "findings": findings,
        "warnings": warnings,
        "recommendations": make_actions("traceability", findings, warnings),
    }


def review_ui_match(documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """기능과 UI 간 일치성 점검."""
    feature_rows = get_rows(get_document(documents, "feature_definition"))
    ui_rows = get_rows(get_document(documents, "ui_design"))

    screen_col = find_column(feature_rows, ["화면id"])
    ui_screen_col = find_column(ui_rows, ["화면id"])
    name_col = find_column(feature_rows, ["기능명"])
    ui_text = " ".join(flatten_row_text(ui_rows)).lower()

    findings: List[str] = []
    warnings: List[str] = []

    if not screen_col:
        findings.append("기능 정의서에서 화면ID 컬럼을 찾지 못했습니다.")
    if not ui_screen_col:
        findings.append("UI 설계서에서 화면ID 컬럼을 찾지 못했습니다.")
    if screen_col and ui_screen_col:
        missing_screens = missing_values(set(non_empty_values(feature_rows, screen_col)), set(non_empty_values(ui_rows, ui_screen_col)))
        if missing_screens:
            findings.append(f"기능 정의서의 화면ID가 UI 설계서에 없습니다. 예: {', '.join(missing_screens[:3])}")
    if name_col:
        low_overlap = []
        for row in feature_rows:
            feature_name = str(row.get(name_col, "")).strip()
            if feature_name and not any(token in ui_text for token in extract_tokens(feature_name)):
                low_overlap.append(feature_name)
        if low_overlap:
            warnings.append(f"UI 문서에서 직접 확인되지 않는 기능명이 있습니다. 예: {', '.join(low_overlap[:2])}")

    score = max(0, 100 - len(findings) * 12 - len(warnings) * 4)
    return {
        "scenario_key": "ui_match",
        "summary": "기능 정의와 UI 설계 간 화면 연결과 용어 일치성을 점검했습니다.",
        "score": score,
        "findings": findings,
        "warnings": warnings,
        "recommendations": make_actions("ui_match", findings, warnings),
    }


def review_coverage(documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """요구사항 대비 기능 완전성 점검."""
    requirement_rows = get_rows(get_document(documents, "requirement_definition"))
    feature_rows = get_rows(get_document(documents, "feature_definition"))

    req_id_col = find_column(requirement_rows, ["요구사항 id"])
    feat_req_col = find_column(feature_rows, ["요구사항 id"])
    req_ids = set(non_empty_values(requirement_rows, req_id_col))
    feat_req_ids = set(non_empty_values(feature_rows, feat_req_col))

    findings: List[str] = []
    warnings: List[str] = []

    if missing_values(req_ids, feat_req_ids):
        findings.append(f"요구사항 대비 기능 정의가 누락된 항목이 있습니다. 예: {', '.join(missing_values(req_ids, feat_req_ids)[:3])}")
    if len(feature_rows) < len(requirement_rows):
        warnings.append("기능 정의 행 수가 요구사항 수보다 적어 세부 기능 분해가 부족할 수 있습니다.")

    score = max(0, 100 - len(findings) * 12 - len(warnings) * 4)
    return {
        "scenario_key": "coverage",
        "summary": "요구사항 대비 기능 정의의 누락과 과잉을 분석했습니다.",
        "score": score,
        "findings": findings,
        "warnings": warnings,
        "recommendations": make_actions("coverage", findings, warnings),
    }


def make_actions(scenario_key: str, findings: List[str], warnings: List[str]) -> List[str]:
    """이슈를 간단한 개선 액션으로 바꿉니다."""
    guide = {
        "basic_quality": "필수 컬럼과 ID 형식을 먼저 정리하세요.",
        "traceability": "요구사항-기능-UI 매핑표를 기준 문서로 고정하세요.",
        "ui_match": "기능명과 화면ID를 1:1 기준으로 다시 매핑하세요.",
        "coverage": "요구사항을 기능 단위로 다시 분해해 누락 항목을 보완하세요.",
    }.get(scenario_key, "핵심 이슈부터 순서대로 보완하세요.")

    actions = []
    for text in findings[:2] + warnings[:2]:
        action = f"{shorten(text)} -> {guide}"
        if action not in actions:
            actions.append(action)
    return actions or ["정기 점검을 유지하세요."]


def get_document(documents: List[Dict[str, Any]], document_key: str) -> Optional[Dict[str, Any]]:
    """문서 키로 문서를 찾습니다."""
    for document in documents:
        if document.get("document_key") == document_key:
            return document
    return None


def get_rows(document: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """문서의 모든 시트 데이터를 하나의 row 목록으로 합칩니다."""
    if not document:
        return []
    rows: List[Dict[str, Any]] = []
    for sheet in document.get("content_summary", {}).get("sheets", []):
        rows.extend(sheet.get("data", []))
    return rows


def get_columns(document: Optional[Dict[str, Any]]) -> List[str]:
    """문서의 컬럼 이름을 중복 없이 합칩니다."""
    if not document:
        return []
    columns: List[str] = []
    for sheet in document.get("content_summary", {}).get("sheets", []):
        for column in sheet.get("columns", []):
            if column and column not in columns:
                columns.append(column)
    return columns


def find_column(rows: List[Dict[str, Any]], keywords: List[str]) -> Optional[str]:
    """행 데이터에서 키워드와 맞는 컬럼을 찾습니다."""
    if not rows:
        return None
    for column in rows[0].keys():
        normalized_column = normalize(column)
        for keyword in keywords:
            if normalize(keyword) in normalized_column:
                return column
    return None


def non_empty_values(rows: List[Dict[str, Any]], column_name: Optional[str]) -> List[str]:
    """선택한 컬럼의 빈 값이 아닌 항목만 반환합니다."""
    if not column_name:
        return []
    return [str(row.get(column_name, "")).strip() for row in rows if str(row.get(column_name, "")).strip()]


def missing_values(source: set, target: set) -> List[str]:
    """source에는 있지만 target에는 없는 값을 정렬해 반환합니다."""
    return sorted(source - target)


def flatten_row_text(rows: List[Dict[str, Any]]) -> List[str]:
    """행 값을 한 줄 텍스트로 펼칩니다."""
    return [" ".join(str(value).strip() for value in row.values() if str(value).strip()) for row in rows]


def extract_tokens(text: str) -> List[str]:
    """문자열에서 비교용 토큰만 단순 추출합니다."""
    return [token.lower() for token in re.findall(r"[A-Za-z0-9가-힣]+", text) if len(token) >= 2]


def normalize(value: Any) -> str:
    """문자열을 비교용으로 정규화합니다."""
    return re.sub(r"\s+", " ", str(value or "").replace("\n", " ").strip().lower())


def shorten(text: str, max_length: int = 30) -> str:
    """긴 문장을 짧은 제목처럼 줄입니다."""
    cleaned = str(text).replace("`", "").strip()
    return cleaned if len(cleaned) <= max_length else f"{cleaned[:max_length - 1]}…"


def parse_json(payload_json: str) -> Dict[str, Any]:
    """문자열 JSON을 dict로 변환합니다."""
    try:
        parsed = json.loads(payload_json)
        return parsed if isinstance(parsed, dict) else {"payload": parsed}
    except json.JSONDecodeError:
        return {"raw_payload": payload_json}


def sanitize_name(name: str) -> str:
    """저장용 파일 이름으로 정리합니다."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_") or "agent"


def save_json(file_path: Path, payload: Dict[str, Any]) -> None:
    """UTF-8 JSON 파일로 저장합니다."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


In [53]:
orchestrator_response = run_orchestrator(agent_request)  # 현재는 Orchestrator 스텁 호출

In [65]:
subagents = []
for spec in SUBAGENT_SPECS: # 서브 에이전트 목록 정의 
    subagents.append(
    {
        "name": spec["name"],
        "description": spec["description"],
        "system_prompt": spec["system_prompt"],
        "tools": [],
        "skills": [],
        "response_format": spec["response_format"],
    }
)

for index, value in enumerate(subagents):
    print(value)

{'name': 'guide-agent', 'description': '시나리오 기준, 필수 문서, 사전 체크를 짧게 정리한다.', 'system_prompt': '너는 Guide Agent다. 시나리오 정의와 현재 문서 현황을 먼저 확인하고, 필수 문서 누락 여부와 체크리스트를 짧게 정리하라.', 'tools': [], 'skills': [], 'response_format': <class '__main__.SubagentReport'>}
{'name': 'qa-agent', 'description': '문서가 애매할 때 추가 확인 질문을 짧게 정리한다.', 'system_prompt': '너는 QA Agent다. 문서 행 수, 파싱 상태, 샘플 행을 보고 추가로 확인해야 할 질문만 간단히 정리하라.', 'tools': [], 'skills': [], 'response_format': <class '__main__.SubagentReport'>}
{'name': 'validation-agent', 'description': 'basic_quality와 traceability를 룰 기반으로 점검한다.', 'system_prompt': '너는 Validation Agent다. basic_quality와 traceability 시나리오에서는 제공된 review tool을 사용해 findings, warnings, score를 정리하라.', 'tools': [], 'skills': [], 'response_format': <class '__main__.SubagentReport'>}
{'name': 'review-agent', 'description': 'ui_match와 coverage를 심층 검토한다.', 'system_prompt': '너는 Review Agent다. ui_match와 coverage 시나리오에서 문서 간 불일치와 누락 위험을 정리하라.', 'tools': [], 'skills': [], 'response_format': <class '__ma